# plb_bench — Normalize & Score

Two-stage workflow:

### Stage 1 — Normalize
Take any producer's raw output (AF3, Protenix, Chai, DynamicBind) and emit a
clean, uniform per-model tree:

```
data/
  proteinx/
    1abc/
      model_000.cif     ← ranked highest-confidence first
      model_001.cif
      manifest.json     ← ranking metadata + raw scores
    6dql/
      model_000.cif
      ...
  af3/
    <different PDBs if the model was only run on some>
  chai/
    ...
```

Each `<producer>/` subtree is self-contained. Different producers can cover
different PDB sets — that's the whole point of the model-first layout.

### Stage 2 — Score
Point OpenStructure at the normalized tree. Every model is already ranked,
cleaned, and OST-compatible, so scoring is pure I/O + metric computation.

### Why two stages?
- **Runs independently per producer.** Normalize Protenix today, AF3 next month.
- **Re-normalize without rescoring** (or vice versa) when you update one
  producer's raw output.
- **Debuggable intermediate state.** You can visually inspect `manifest.json`
  before committing CPU hours to OST.

## 0 · Setup

In [ ]:
import sys, subprocess
from pathlib import Path

# Locate the plb_bench package
def _find_package():
    try:
        import plb_bench  # noqa
        return None
    except ImportError:
        pass
    candidates = []
    p = Path.cwd()
    for _ in range(4):
        candidates.extend([p, p / "plb_bench"])
        p = p.parent
    for c in candidates:
        if (c / "plb_bench" / "__init__.py").exists():
            return c
    return "MISSING"

_loc = _find_package()
if _loc == "MISSING":
    raise RuntimeError(
        "plb_bench not found. Install with `pip install -e /path/to/plb_bench` "
        "or set sys.path manually."
    )
if _loc is not None:
    sys.path.insert(0, str(_loc))
    print(f"Added to sys.path: {_loc}")
else:
    print("plb_bench already importable")

# Check dependencies
for pkg, name in [("numpy", "numpy"), ("pandas", "pandas"), ("pyarrow", "pyarrow"),
                   ("requests", "requests"), ("biopython", "Bio"),
                   ("gemmi", "gemmi"), ("rdkit", "rdkit")]:
    try:
        __import__(name)
    except ImportError:
        print(f"⚠ Missing: {pkg}  →  pip install {pkg}")

from plb_bench import DISCOVERERS
print(f"\nRegistered discoverers: {sorted(DISCOVERERS.keys())}")

try:
    import ost
    print(f"✓ OpenStructure {ost.__version__}  (scoring will work)")
except ImportError:
    print("⚠ OpenStructure not installed  (Stage 1 still works; Stage 2 needs it)")
    print("   Install:  conda install -c bioconda openstructure")

## 1 · Paths

Edit these once. Everything downstream reads from them.

In [ ]:
# Top-level paths
DATA_ROOT   = Path("/path/to/data")              # Stage-1 output root; Stage-2 input root
REFS_DIR    = Path("/path/to/refs_cache")        # reference mmCIFs (RCSB download cache)
OUTPUT_DIR  = Path("/path/to/benchmark_output")  # Stage-2 scoring table + artifacts

# Raw producer directories
# Map of producer name → the producer's top-level raw directory.
# Each raw directory should contain per-PDB subdirectories (named by pdb_id).
# Only list the producers you actually have data for.
RAW_ROOTS = {
    "proteinx": Path("/path/to/raw/proteinx"),
    # "af3":         Path("/path/to/raw/af3"),
    # "chai":        Path("/path/to/raw/chai"),
    # "dynamicbind": Path("/path/to/raw/dynamicbind"),
}

# Create output paths
DATA_ROOT.mkdir(parents=True, exist_ok=True)
REFS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Verify raw roots
print("Raw roots:")
for producer, raw in RAW_ROOTS.items():
    exists = raw.exists() and raw.is_dir()
    n_pdb = sum(1 for p in raw.iterdir() if p.is_dir()) if exists else 0
    if exists:
        print(f"  ✓ {producer}: {raw}  ({n_pdb} PDB subdirs)")
    else:
        print(f"  ✗ {producer}: {raw}  (does not exist)")

## 2 · Stage 1 — Normalize raw outputs

For each producer in `RAW_ROOTS`, walk its raw directory, extract confidence
metadata, rank by confidence, and emit
`DATA_ROOT/<producer>/<pdb_id>/model_NNN.cif` + `manifest.json`.

Each (producer, pdb_id) is processed independently. A corrupt CIF in one PDB
doesn't block other PDBs — the bad model just gets `status: sanitize_failed`
in its manifest.

In [ ]:
from plb_bench.normalize import normalize_producer
import pandas as pd

all_entries = []
for producer, raw_root in RAW_ROOTS.items():
    if not raw_root.exists():
        print(f"⊘ Skipping {producer}: {raw_root} does not exist")
        continue
    print(f"\n▶ Normalizing {producer} ...")
    entries = normalize_producer(producer, raw_root, DATA_ROOT, overwrite=True)
    for e in entries:
        all_entries.append({
            "producer": e.producer,
            "pdb_id": e.pdb_id,
            "n_models": e.n_models,
            "failed": e.failed,
            "best_confidence": e.best_confidence,
            "output_dir": str(e.output_dir.relative_to(DATA_ROOT)),
        })
    print(f"  {len(entries)} (producer, pdb_id) groups")
    print(f"  Total models: {sum(e.n_models for e in entries)}, "
          f"failed: {sum(e.failed for e in entries)}")

normalization_summary = pd.DataFrame(all_entries)
if len(normalization_summary):
    print(f"\nTotal: {len(normalization_summary)} groups, "
          f"{normalization_summary['n_models'].sum()} models OK, "
          f"{normalization_summary['failed'].sum()} failures")
normalization_summary

In [ ]:
# Inspect one manifest to see what was captured
if len(normalization_summary):
    import json
    sample_row = normalization_summary.iloc[0]
    manifest_path = DATA_ROOT / sample_row["producer"] / sample_row["pdb_id"] / "manifest.json"
    manifest = json.loads(manifest_path.read_text())
    print(f"Manifest: {manifest_path}")
    print(f"  producer: {manifest['producer']}")
    print(f"  pdb_id:   {manifest['pdb_id']}")
    print(f"  n_models: {manifest['n_models']} (ok: {manifest['n_ok']}, failed: {manifest['n_failed']})")
    print(f"\nModel 0 (top-ranked):")
    top = next(m for m in manifest["models"] if m["model_idx"] == 0)
    for k, v in top.items():
        if k == "raw_scores":
            print(f"  raw_scores: {v}")
        elif isinstance(v, (int, float, str, bool)) or v is None:
            print(f"  {k}: {v}")

In [ ]:
# Show the normalized tree shape
print(f"Contents of {DATA_ROOT}:\n")
for producer_dir in sorted(DATA_ROOT.iterdir()):
    if not producer_dir.is_dir():
        continue
    pdbs = sorted(p for p in producer_dir.iterdir() if p.is_dir())
    print(f"  {producer_dir.name}/  ({len(pdbs)} PDB entries)")
    for pdb_dir in pdbs[:3]:
        cifs = sorted(pdb_dir.glob("model_*.cif"))
        print(f"    {pdb_dir.name}/  ({len(cifs)} models)")
        for c in cifs[:3]:
            print(f"      {c.name}  ({c.stat().st_size:,} bytes)")
        if len(cifs) > 3:
            print(f"      ... and {len(cifs) - 3} more")
    if len(pdbs) > 3:
        print(f"    ... and {len(pdbs) - 3} more PDBs")

## 3 · Reference availability

Which `pdb_id`s have references cached locally vs. need downloading. Pre-warm
the cache here if you want the scoring step to be purely compute-bound.

In [ ]:
from plb_bench.references import get_reference

PREWARM = False   # set True to download all missing references now

pdb_ids = sorted({e["pdb_id"] for e in all_entries})
ref_rows = []
for pid in pdb_ids:
    try:
        path, source = get_reference(pid, REFS_DIR, allow_download=False)
        ref_rows.append({"pdb_id": pid, "status": "cached",
                         "source": source, "file": path.name})
    except FileNotFoundError:
        if PREWARM:
            try:
                path, source = get_reference(pid, REFS_DIR, allow_download=True)
                ref_rows.append({"pdb_id": pid, "status": "downloaded",
                                 "source": source, "file": path.name})
            except Exception as e:
                ref_rows.append({"pdb_id": pid, "status": "UNAVAILABLE",
                                 "source": "—", "file": f"{type(e).__name__}: {e}"})
        else:
            ref_rows.append({"pdb_id": pid, "status": "missing (will download)",
                             "source": "rcsb", "file": "—"})

refs_summary = pd.DataFrame(ref_rows)
if len(refs_summary):
    print(refs_summary["status"].value_counts().to_string())
refs_summary

## 4 · Stage 2 — Score with OpenStructure

Reads the normalized tree. Computes **BiSyRMSD**, **lDDT-PLI**, and
**QS-global** for every `model_NNN.cif` against its reference. Work is
parallelized across `(producer, pdb_id)` groups.

Set `SCORE_PRODUCERS` or `SCORE_PDB_IDS` to a list to score a subset, or
leave `None` to score everything.

In [ ]:
from plb_bench.score_tree import ScoreConfig, run as score_run

SCORE_PRODUCERS        = None      # e.g. ["proteinx"] to score just one
SCORE_PDB_IDS          = None      # e.g. ["1abc", "6dql"] to score a slice
N_WORKERS              = None      # None → os.cpu_count()
DOWNLOAD_MISSING_REFS  = True
USE_RAY                = False
OUTPUT_FORMAT          = "parquet"

cfg = ScoreConfig(
    normalized_root=DATA_ROOT,
    refs_dir=REFS_DIR,
    output_dir=OUTPUT_DIR,
    producers=SCORE_PRODUCERS,
    pdb_ids=SCORE_PDB_IDS,
    n_workers=N_WORKERS,
    use_ray=USE_RAY,
    download_missing_refs=DOWNLOAD_MISSING_REFS,
    output_format=OUTPUT_FORMAT,
)

output_path = score_run(cfg)
print(f"\n✓ Results written to: {output_path}")

## 5 · Load the benchmark table

In [ ]:
if output_path.suffix == ".parquet":
    df = pd.read_parquet(output_path)
else:
    df = pd.read_csv(output_path)

print(f"Rows: {len(df)}")
print(f"Columns: {list(df.columns)}")
print(f"\nStatus distribution:")
print(df["status"].value_counts().to_string())
df.head(10)

## 6 · Inspect failures

In [ ]:
failures = df[df["status"] != "ok"]
if len(failures) == 0:
    print("No failures — all models scored successfully.")
else:
    print(f"{len(failures)} failing rows:")
    display(failures[["pdb_id", "producer", "model_idx", "status", "error"]])

## 7 · Per-producer summary

In [ ]:
ok = df[df["status"] == "ok"]

per_producer = (
    ok.groupby("producer")
      .agg(
          n_scored=("model_idx", "count"),
          n_pdbs=("pdb_id", "nunique"),
          mean_bisy_rmsd=("bisy_rmsd", "mean"),
          median_bisy_rmsd=("bisy_rmsd", "median"),
          mean_lddt_pli=("lddt_pli", "mean"),
          mean_qs_global=("qs_global", "mean"),
          mean_confidence=("confidence", "mean"),
      )
      .round(3)
)
per_producer

## 8 · Best-pose comparison

`model_idx = 0` only — each producer's self-picked top prediction, pivoted
to make head-to-head comparison easy.

Cells are `NaN` for (producer, pdb_id) pairs that weren't run — expected
when different producers cover different PDB sets.

In [ ]:
best = (
    ok[ok["model_idx"] == 0]
    .pivot_table(
        index="pdb_id",
        columns="producer",
        values=["bisy_rmsd", "lddt_pli", "qs_global"],
    )
    .round(3)
)
best

## 9 · Confidence calibration

For each producer, how well does its self-reported confidence predict
pose quality?

- **negative** `corr_conf_vs_bisy_rmsd` is good (high confidence → low RMSD)
- **positive** `corr_conf_vs_lddt_pli` is good (high confidence → high lDDT)

In [ ]:
def _correlations(g):
    return pd.Series({
        "n": len(g),
        "corr_conf_vs_bisy_rmsd": g["confidence"].corr(g["bisy_rmsd"]),
        "corr_conf_vs_lddt_pli":  g["confidence"].corr(g["lddt_pli"]),
        "corr_conf_vs_qs_global": g["confidence"].corr(g["qs_global"]),
    })

try:
    calib = ok.groupby("producer").apply(_correlations, include_groups=False)
except TypeError:
    calib = ok.groupby("producer").apply(_correlations)
calib.round(3)

## 10 · Re-running one piece

The two-stage design lets you touch pieces independently:

**Re-normalize only one producer** (new AF3 run arrived):
```python
from plb_bench.normalize import normalize_producer
normalize_producer("af3", Path("/path/to/raw/af3"), DATA_ROOT, overwrite=True)
```

**Rescore only one producer** (say, updated references):
```python
cfg = ScoreConfig(normalized_root=DATA_ROOT, refs_dir=REFS_DIR,
                   output_dir=OUTPUT_DIR, producers=["af3"])
score_run(cfg)
```

**Rescore specific PDB IDs**:
```python
cfg = ScoreConfig(normalized_root=DATA_ROOT, refs_dir=REFS_DIR,
                   output_dir=OUTPUT_DIR, pdb_ids=["1abc", "6dql"])
score_run(cfg)
```

**Add a new producer**:
```python
from plb_bench.normalize import register_discoverer
from plb_bench.schema import ModelRecord

@register_discoverer("mynewmodel")
def discover_mynewmodel(raw_root):
    for pdb_dir in sorted(raw_root.iterdir()):
        if not pdb_dir.is_dir():
            continue
        records = [...]   # build ModelRecord per model with .confidence
        yield pdb_dir.name.lower(), records
```
Then add `"mynewmodel": Path("/path/to/raw/mynewmodel")` to `RAW_ROOTS`
and rerun Stage 1.